<a href="https://colab.research.google.com/github/jayvazil/datasciencepracticumgrp3/blob/main/stream_lit_malaria_prediction_using_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║                        CELL 1 — INSTALLATION                    ║
# ║  This cell installs all Python libraries needed by the app.     ║
# ║                            ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── streamlit: the web app framework that builds the browser UI ───
# ── scikit-learn: machine learning library (Random Forest, etc.) ──
# ── joblib: saves/loads trained models to disk so they persist ────
# ── pandas, numpy, matplotlib come pre-installed in Colab already ─
!pip install streamlit scikit-learn joblib -q

print("✅ All packages installed successfully")


✅ All packages installed successfully


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║                  CELL 2 — THE STREAMLIT APP                     ║
# ║  %%writefile writes everything below it into malaria_app.py     ║
# ║  on the Colab disk. It does NOT run the app — Cell 3 does that. ║
# ║   ║
# ╚══════════════════════════════════════════════════════════════════╝
%%writefile malaria_app.py

# ══════════════════════════════════════════════════════════════════
# SECTION 1 — LIBRARY IMPORTS
# Here we bring in every tool the app needs:
#   - streamlit  : builds all the buttons, sliders, charts in browser
#   - pandas     : loads and manipulates the CSV dataset as a table
#   - numpy      : fast number crunching (arrays, math operations)
#   - matplotlib : draws all the charts (bar charts, ROC curves etc.)
#   - joblib     : saves trained models to disk so sliders don't lose them
#   - os         : interacts with the file system (create folders etc.)
#   - warnings   : suppresses noisy non-critical warning messages
#   - sklearn    : the machine learning library — contains:
#       train_test_split        → splits data into training and test sets
#       StandardScaler          → normalises features (mean=0, std=1)
#       LabelEncoder            → converts text categories to numbers
#       LogisticRegression      → Model 1: simple linear classifier
#       RandomForestClassifier  → Model 2: ensemble of decision trees
#       GradientBoostingClassifier → Model 3: boosted tree ensemble
#       SimpleImputer           → fills in missing values with median
#       accuracy_score etc.     → measures how good each model is
# ══════════════════════════════════════════════════════════════════
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, roc_curve)


# ══════════════════════════════════════════════════════════════════
# SECTION 2 — STREAMLIT PAGE CONFIGURATION
# This is where Streamlit starts. set_page_config() must be the
# very first Streamlit call in the script.
#   - page_title : text shown on the browser tab
#   - page_icon  : emoji shown on the browser tab
#   - layout     : "wide" uses the full browser width
# The st.markdown() calls inject raw HTML/CSS to style the table
# and draw the green/teal gradient banner at the top of the page.
# ══════════════════════════════════════════════════════════════════
st.set_page_config(page_title="Malaria Prediction | Group 3", page_icon="🦟", layout="wide")

# CSS override — forces table text to always be black on white
# (Streamlit's default dark theme can make table text invisible)
st.markdown("""
<style>
table {color:#000000 !important; background:#ffffff !important;}
th {background:#1a5276 !important; color:#ffffff !important; padding:10px !important;}
td {color:#000000 !important; background:#ffffff !important; padding:8px !important;}
tr:nth-child(even) td {background:#eaf4fb !important;}
</style>
""", unsafe_allow_html=True)

# Hero banner — rendered as raw HTML inside the Streamlit page
st.markdown("""
<div style="background:linear-gradient(135deg,#1a5276,#117a65);padding:2rem;border-radius:12px;color:white;margin-bottom:1rem">
<h1>🦟 Malaria Infection Prediction System</h1>
<p>Group 3 · BSc Data Science · Meru University of Science and Technology · 2026<br>
<em>Nyanza · Rift Valley · Central Kenya</em></p>
</div>
""", unsafe_allow_html=True)


# ══════════════════════════════════════════════════════════════════
# SECTION 3 — GLOBAL CONSTANTS
# Defined once here so they are reused consistently everywhere:
#   FEATURES     : the 12 input columns fed into the ML models
#   METRICS      : the 5 evaluation scores shown in the results table
#   COLORS       : chart colours for the 3 models (blue, green, orange)
#   MN           : month name labels for the selectbox widget
#   MODELS_AVAIL : the 3 model names shown in the sidebar checkbox
#   SAVE_DIR     : folder on Colab disk where trained models are saved
#                  /tmp is used because it persists across Streamlit
#                  reruns within the same Colab session
# ══════════════════════════════════════════════════════════════════
FEATURES = ['Rainfall_mm','Temperature_C','Humidity_percent','Lag_1_Month_Cases',
            'Incidence_per_100k','Month','Population','Malaria_Cases',
            'Cases_Per_Capita','Region_enc','County_enc','Season_enc']
METRICS  = ['Accuracy','Precision','Recall','F1 Score','ROC-AUC']
COLORS   = ['#1a5276','#117a65','#d35400']
MN       = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
MODELS_AVAILABLE = ["Logistic Regression","Random Forest","Gradient Boosting"]
SAVE_DIR = "/tmp/malaria_models"

# Create the save folder if it doesn't already exist
os.makedirs(SAVE_DIR, exist_ok=True)


# ══════════════════════════════════════════════════════════════════
# SECTION 4 — SIDEBAR: FILE UPLOAD
# st.sidebar.* puts widgets in the left panel instead of main page.
# file_uploader() creates a drag-and-drop box that accepts CSV files.
# The user must upload their dataset here before anything else works.
# ══════════════════════════════════════════════════════════════════
st.sidebar.header("📂 Dataset")
uploaded = st.sidebar.file_uploader("Upload CSV", type=["csv"])


# ══════════════════════════════════════════════════════════════════
# SECTION 5 — DATA LOADING
# @st.cache_data tells Streamlit to cache (remember) the result of
# this function. If the same file is uploaded again, it won't
# reload it from disk — it reuses the cached result. This makes
# the app much faster when sliders are moved.
# pd.read_csv() reads the CSV file into a pandas DataFrame (table).
# ══════════════════════════════════════════════════════════════════
@st.cache_data
def load_data(file):
    return pd.read_csv(file)

# Gate: if no file is uploaded yet, show a hint and stop execution.
# st.stop() halts the script here — nothing below runs until a file
# is uploaded.
if not uploaded:
    st.info("⬅️ Upload Final_Malaria_Dataset.csv in the sidebar to begin.")
    st.stop()

df_raw = load_data(uploaded)


# ══════════════════════════════════════════════════════════════════
# SECTION 6 — DATA PREPROCESSING (CLEANING & FEATURE ENGINEERING)
# This function transforms the raw CSV into clean model-ready data.
# Also cached so it only runs once per uploaded file.
#
# Steps inside preprocess():
#   1. DROP IRRELEVANT COLUMNS
#      Columns like ID, Notes, Health_Facilities carry no predictive
#      signal for malaria risk, so they are removed.
#
#   2. REMOVE DUPLICATE ROWS
#      drop_duplicates() removes any rows that appear more than once,
#      preventing the model from overfitting to repeated data.
#
#   3. STANDARDISE TEXT COLUMNS
#      .str.strip() removes leading/trailing spaces.
#      .str.title() makes "NYANZA" → "Nyanza" (consistent casing).
#
#   4. HANDLE MISSING VALUES (IMPUTATION)
#      SimpleImputer fills any NaN cells with the median of that
#      column. Using median (not mean) is more robust to outliers.
#
#   5. REMOVE OUTLIERS (IQR CLIPPING)
#      For each numeric column, values below Q1-1.5*IQR or above
#      Q3+1.5*IQR are clipped to those boundary values. This stops
#      extreme values from distorting the model.
#
#   6. FEATURE ENGINEERING — SEASON
#      A new 'Season' column is derived from the Month number:
#        Mar-May  → Long Rains (peak malaria season)
#        Jun-Aug  → Dry
#        Sep-Nov  → Short Rains
#        Dec-Feb  → Cool Dry
#
#   7. FEATURE ENGINEERING — CASES PER CAPITA
#      Malaria_Cases / Population * 100000 gives a rate that is
#      comparable across counties of different population sizes.
#
#   8. LABEL ENCODING
#      ML models only accept numbers, not text. LabelEncoder
#      converts Region, County, Season to integer codes.
#      e.g. "Nyanza" → 1, "Rift Valley" → 2
#
#   9. RETURN ONLY THE 12 MODEL FEATURES + TARGET
#      X  = the 12 input features (what the model learns from)
#      y  = High_Risk_Binary (0=Low Risk, 1=High Risk) — what we predict
# ══════════════════════════════════════════════════════════════════
@st.cache_data
def preprocess(df):
    df = df.copy()

    # Step 1: Drop columns with no predictive value
    for c in ['ID','Health_Facilities','Avg_Income','Disease_Cases','Notes']:
        if c in df.columns:
            df.drop(columns=c, inplace=True)

    # Step 2: Remove duplicate rows
    df.drop_duplicates(inplace=True)

    # Step 3: Clean text columns — strip spaces and fix capitalisation
    df['Region'] = df['Region'].str.strip().str.title()
    df['County'] = df['County'].str.strip().str.title()

    # Step 4: Impute (fill) missing numeric values with column median
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    df[num_cols] = SimpleImputer(strategy='median').fit_transform(df[num_cols])

    # Step 5: Clip outliers using the IQR method for key numeric columns
    for col in ['Rainfall_mm','Temperature_C','Humidity_percent',
                'Malaria_Cases','Lag_1_Month_Cases','Incidence_per_100k']:
        if col in df.columns:
            Q1, Q3 = df[col].quantile([0.25, 0.75])
            IQR    = Q3 - Q1
            df[col] = df[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

    # Step 6: Create Season feature from Month number
    def season(m):
        return ('Long_Rains'  if m in [3,4,5]   else
                'Dry'         if m in [6,7,8]   else
                'Short_Rains' if m in [9,10,11] else 'Cool_Dry')
    df['Season'] = df['Month'].apply(season)

    # Step 7: Cases per capita — normalises case counts by population
    df['Cases_Per_Capita'] = df['Malaria_Cases'] / df['Population'] * 100000

    # Step 8: Encode categorical text columns into integer numbers
    df['Region_enc'] = LabelEncoder().fit_transform(df['Region'])
    df['County_enc'] = LabelEncoder().fit_transform(df['County'])
    df['Season_enc'] = LabelEncoder().fit_transform(df['Season'])

    # Step 9: Return the 12 features (X) and the binary target (y)
    return df[FEATURES], df['High_Risk_Binary'].astype(int)

# Run preprocessing on the uploaded data
X, y = preprocess(df_raw)


# ══════════════════════════════════════════════════════════════════
# SECTION 7 — DATASET OVERVIEW METRICS (STREAMLIT DISPLAY)
# st.columns(5) creates 5 equal-width columns side by side.
# .metric() displays a big number card with a label above it.
# These give the user a quick summary of the loaded dataset.
# ══════════════════════════════════════════════════════════════════
st.subheader("📊 Dataset Overview")
c1,c2,c3,c4,c5 = st.columns(5)
c1.metric("Records",        f"{len(X):,}")
c2.metric("Features",       len(FEATURES))
c3.metric("Low Risk",       f"{int((y==0).sum()):,}")
c4.metric("High Risk",      f"{int((y==1).sum()):,}")
c5.metric("High-Risk Rate", f"{y.mean()*100:.1f}%")


# ══════════════════════════════════════════════════════════════════
# SECTION 8 — SIDEBAR: MODEL SETTINGS
# These widgets let the user configure training before clicking Train:
#   slider    → choose what % of data to hold out for testing
#               e.g. 20% = 80% trains the model, 20% evaluates it
#   multiselect → choose which of the 3 models to train
#                 defaults to all 3 selected
# ══════════════════════════════════════════════════════════════════
st.sidebar.header("⚙️ Settings")
test_size  = st.sidebar.slider("Test split %", 10, 40, 20) / 100
models_sel = st.sidebar.multiselect("Models", MODELS_AVAILABLE, default=MODELS_AVAILABLE)

if not models_sel:
    st.warning("Select at least one model."); st.stop()


# ══════════════════════════════════════════════════════════════════
# SECTION 9 — MODEL TRAINING
# Triggered when the user clicks "Train Models" in the sidebar,
# OR automatically if no saved models exist yet.
#
# Steps:
#   TRAIN/TEST SPLIT
#     train_test_split() divides X and y into:
#       X_tr / y_tr → training set (model learns from this)
#       X_te / y_te → test set    (model is evaluated on this)
#     stratify=y ensures both sets have the same ratio of 0s and 1s.
#     random_state=42 makes the split reproducible every time.
#
#   FEATURE SCALING (StandardScaler)
#     Logistic Regression is sensitive to feature scale, so we
#     normalise: subtract mean, divide by std deviation.
#     This makes all features sit on a similar numeric scale.
#     fit_transform() on training data → learns mean/std from training
#     transform()     on test data     → applies the SAME scaling
#     (IMPORTANT: never fit on test data — that would be data leakage)
#     Random Forest and Gradient Boosting are tree-based — they are
#     NOT sensitive to scale, so raw X_tr/X_te are used for them.
#
#   THE 3 MODELS:
#     Logistic Regression   — linear boundary, fast, interpretable
#     Random Forest         — 200 decision trees voting together
#     Gradient Boosting     — 200 trees each correcting the last one
#
#   EVALUATION METRICS:
#     Accuracy  = (correct predictions) / (total predictions)
#     Precision = of all predicted High-Risk, how many were actually HR?
#     Recall    = of all actual High-Risk, how many did we catch?
#     F1 Score  = harmonic mean of Precision and Recall (best balance)
#     ROC-AUC   = area under ROC curve (1.0 = perfect, 0.5 = random)
#
#   SAVING WITH JOBLIB
#     Each trained model is saved to /tmp/malaria_models/<name>.pkl
#     The scaler is saved too — it must be used again at prediction time.
#     Performance metrics are saved as a dictionary in perf.pkl
#     This is what makes sliders work: models survive Streamlit reruns.
# ══════════════════════════════════════════════════════════════════
models_trained = all(
    os.path.exists(f"{SAVE_DIR}/{n.replace(' ','_')}.pkl") for n in models_sel
)

if st.sidebar.button("🚀 Train Models", type="primary") or not models_trained:
    st.subheader("🔧 Training Models...")

    # ── TRAIN / TEST SPLIT ────────────────────────────────────
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    # ── FEATURE SCALING (for Logistic Regression only) ────────
    sc      = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)   # fit on training, then transform
    X_te_sc = sc.transform(X_te)       # only transform test (no fitting)
    joblib.dump(sc, f"{SAVE_DIR}/scaler.pkl")  # save scaler to disk

    perf = {}   # dictionary to store each model's evaluation results
    prog = st.progress(0)  # progress bar widget (0% to 100%)

    for i, name in enumerate(models_sel):
        st.write(f"⏳ Training {name}...")

        # ── MODEL 1: LOGISTIC REGRESSION ──────────────────────
        # A linear model that estimates probability using a sigmoid
        # function. Uses the SCALED data (X_tr_sc / X_te_sc).
        # max_iter=1000 allows enough iterations to converge.
        if name == "Logistic Regression":
            m = LogisticRegression(max_iter=1000, random_state=42)
            m.fit(X_tr_sc, y_tr)
            yp  = m.predict(X_te_sc)        # class predictions (0 or 1)
            ypr = m.predict_proba(X_te_sc)[:,1]  # probability of class 1

        # ── MODEL 2: RANDOM FOREST ────────────────────────────
        # Builds 200 independent decision trees on random subsets
        # of the data (bagging). Final prediction = majority vote.
        # Uses RAW (unscaled) data — trees don't need scaling.
        elif name == "Random Forest":
            m = RandomForestClassifier(n_estimators=200, random_state=42)
            m.fit(X_tr, y_tr)
            yp  = m.predict(X_te)
            ypr = m.predict_proba(X_te)[:,1]

        # ── MODEL 3: GRADIENT BOOSTING ────────────────────────
        # Builds 200 trees sequentially — each tree corrects the
        # errors of the previous one (boosting). Also uses RAW data.
        else:
            m = GradientBoostingClassifier(n_estimators=200, random_state=42)
            m.fit(X_tr, y_tr)
            yp  = m.predict(X_te)
            ypr = m.predict_proba(X_te)[:,1]

        # ── CALCULATE EVALUATION METRICS ──────────────────────
        perf[name] = {
            'Accuracy':  round(accuracy_score(y_te, yp),  4),
            'Precision': round(precision_score(y_te, yp, zero_division=0), 4),
            'Recall':    round(recall_score(y_te,    yp, zero_division=0), 4),
            'F1 Score':  round(f1_score(y_te,        yp, zero_division=0), 4),
            'ROC-AUC':   round(roc_auc_score(y_te,  ypr), 4),
            'yp':  yp.tolist(),               # predicted labels (serialisable)
            'ypr': ypr.tolist(),              # predicted probabilities
            'cm':  confusion_matrix(y_te, yp).tolist(),   # 2x2 matrix
            'fi':  m.feature_importances_.tolist()        # RF/GB only
                   if hasattr(m, 'feature_importances_') else None
        }

        # ── SAVE MODEL TO DISK ────────────────────────────────
        joblib.dump(m, f"{SAVE_DIR}/{name.replace(' ','_')}.pkl")
        prog.progress((i+1) / len(models_sel))  # update progress bar

    # Save performance dict and test labels for later display
    joblib.dump(perf,           f"{SAVE_DIR}/perf.pkl")
    joblib.dump(y_te.tolist(),  f"{SAVE_DIR}/y_te.pkl")
    st.success("✅ All models trained and saved!")


# ══════════════════════════════════════════════════════════════════
# SECTION 10 — LOAD SAVED RESULTS FROM DISK
# On every Streamlit rerun (e.g. slider moved), we reload the saved
# performance data and test labels from /tmp instead of retraining.
# This is what keeps the app fast and the sliders responsive.
# ══════════════════════════════════════════════════════════════════
if not os.path.exists(f"{SAVE_DIR}/perf.pkl"):
    st.info("👈 Click **Train Models** in the sidebar to begin.")
    st.stop()

perf      = joblib.load(f"{SAVE_DIR}/perf.pkl")
y_te_list = joblib.load(f"{SAVE_DIR}/y_te.pkl")
y_te_arr  = np.array(y_te_list)

# Only keep models that were actually trained and have saved results
trained = [n for n in models_sel if n in perf]
if not trained:
    st.info("👈 Click **Train Models** to train the selected models.")
    st.stop()


# ══════════════════════════════════════════════════════════════════
# SECTION 11 — PERFORMANCE SUMMARY TABLE (STREAMLIT HTML TABLE)
# Finds the best model by F1 Score (most balanced metric for
# imbalanced datasets like malaria case data).
# Builds an HTML string row by row and renders it with st.markdown().
# The best model's row is highlighted green (#d5f5e3) with ⭐ Best.
# ══════════════════════════════════════════════════════════════════
st.subheader("📈 Performance Summary")
best = max(trained, key=lambda n: perf[n]['F1 Score'])

rows = ""
for idx, name in enumerate(trained):
    ib  = (name == best)
    bg  = '#d5f5e3' if ib else ('#eaf4fb' if idx%2==0 else '#ffffff')
    fw  = 'bold' if ib else 'normal'
    tag = '  ⭐ Best' if ib else ''
    cells = ''.join(
        f'<td style="padding:10px;border:1px solid #999;color:#000;text-align:center;">'
        f'{perf[name][m]:.4f}</td>' for m in METRICS
    )
    rows += (f'<tr style="background:{bg};">'
             f'<td style="padding:10px;border:1px solid #999;color:#000;font-weight:{fw};">'
             f'{name}{tag}</td>{cells}</tr>')

hdrs = ''.join(
    f'<th style="padding:10px;border:1px solid #999;text-align:center;">{m}</th>'
    for m in METRICS
)
st.markdown(
    f'<table style="width:100%;border-collapse:collapse;font-size:15px;">'
    f'<tr style="background:#1a5276;color:#fff;">'
    f'<th style="padding:10px;border:1px solid #999;text-align:left;">Model</th>'
    f'{hdrs}</tr>{rows}</table>',
    unsafe_allow_html=True
)
st.success(f"🏆 Best Model: **{best}** — F1 = {perf[best]['F1 Score']:.4f}")


# ══════════════════════════════════════════════════════════════════
# SECTION 12 — CHARTS (3 TABS)
# st.tabs() creates clickable tab panels — user switches between them.
#
# TAB 1 — GROUPED BAR CHART (Model Metrics Comparison)
#   Plots all 5 metrics side by side for all trained models.
#   Value labels are printed on top of each bar.
#
# TAB 2 — CONFUSION MATRICES
#   A 2x2 grid showing:
#     True Negative  | False Positive
#     False Negative | True Positive
#   Darker blue = more predictions in that cell.
#   White text on dark cells, black text on light cells.
#
# TAB 3 — ROC CURVES
#   Plots True Positive Rate vs False Positive Rate at every threshold.
#   AUC (Area Under Curve) closer to 1.0 = better model.
#   The dashed diagonal = random guessing baseline (AUC = 0.5).
# ══════════════════════════════════════════════════════════════════
t1, t2, t3 = st.tabs(["📊 Metrics","🟦 Confusion Matrices","📉 ROC Curves"])

with t1:
    # Grouped bar chart — one group per metric, one bar per model
    fig, ax = plt.subplots(figsize=(11,5))
    x = np.arange(len(METRICS))
    w = 0.7 / len(trained)   # bar width shrinks as more models added
    for i, name in enumerate(trained):
        vals = [perf[name][m] for m in METRICS]
        bars = ax.bar(
            x + i*w - (len(trained)-1)*w/2, vals, w,
            label=name, color=COLORS[i%3], alpha=0.88, edgecolor='white'
        )
        # Print numeric value on top of each bar
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(METRICS); ax.set_ylim(0,1.15)
    ax.set_title('Model Performance Comparison', fontweight='bold')
    ax.legend(); ax.spines[['top','right']].set_visible(False)
    plt.tight_layout(); st.pyplot(fig); plt.close()

with t2:
    # One confusion matrix per trained model, side by side
    cols = st.columns(len(trained))
    for i, name in enumerate(trained):
        cm = np.array(perf[name]['cm'])
        fig, ax = plt.subplots(figsize=(4,3.5))
        im = ax.imshow(cm, cmap='Blues'); plt.colorbar(im, ax=ax)
        ax.set_xticks([0,1]); ax.set_yticks([0,1])
        ax.set_xticklabels(['Low','High'], rotation=30)
        ax.set_yticklabels(['Low','High'])
        thresh = cm.max() / 2
        for r in range(2):
            for c in range(2):
                # White text on dark blue cells, black text on light cells
                ax.text(c, r, str(cm[r,c]), ha='center', va='center',
                        fontsize=14, fontweight='bold',
                        color='white' if cm[r,c] > thresh else 'black')
        ax.set_title(name, fontweight='bold')
        plt.tight_layout(); cols[i].pyplot(fig); plt.close()

with t3:
    # ROC curve for each model on the same axes
    fig, ax = plt.subplots(figsize=(8,6))
    for i, name in enumerate(trained):
        fpr, tpr, _ = roc_curve(y_te_arr, np.array(perf[name]['ypr']))
        ax.plot(fpr, tpr, lw=2.5, color=COLORS[i%3],
                label=f'{name} (AUC={perf[name]["ROC-AUC"]:.3f})')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Random baseline')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves', fontweight='bold'); ax.legend()
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout(); st.pyplot(fig); plt.close()


# ══════════════════════════════════════════════════════════════════
# SECTION 13 — FEATURE IMPORTANCE (RANDOM FOREST ONLY)
# Random Forest tracks how much each feature reduces impurity
# across all 200 trees. Higher bar = more important feature.
# Only available for tree-based models (not Logistic Regression).
# Top 25% most important features highlighted in red.
# ══════════════════════════════════════════════════════════════════
if "Random Forest" in trained and perf["Random Forest"]['fi']:
    st.subheader("🌲 Feature Importance (Random Forest)")
    fi = pd.Series(perf['Random Forest']['fi'], index=FEATURES).sort_values()
    fig, ax = plt.subplots(figsize=(9,5))
    ax.barh(fi.index, fi.values,
            color=['#c0392b' if v >= fi.quantile(0.75) else '#1a5276'
                   for v in fi.values],
            alpha=0.88, edgecolor='white')
    ax.set_title('Feature Importance — top features shown in red',
                 fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout(); st.pyplot(fig); plt.close()


# ══════════════════════════════════════════════════════════════════
# SECTION 14 — LIVE PREDICTION PANEL
# This section is the KEY FIX for the slider problem.
#
# WHY SLIDERS USED TO BE BROKEN:
#   Every time a slider is moved, Streamlit reruns the ENTIRE script
#   from top to bottom. Previously, the trained models only lived in
#   the variable `res` in memory — so they were wiped on every rerun.
#   The prediction section would silently fail or use stale values.
#
# HOW IT IS FIXED NOW:
#   Models are loaded fresh from /tmp/malaria_models/ on every rerun.
#   There is NO predict button — the prediction runs automatically
#   every time any slider changes. Results update instantly.
#
# HELPER FUNCTION — enc_season():
#   Converts the numeric month into a season integer code, matching
#   the same encoding used during preprocessing/training.
# ══════════════════════════════════════════════════════════════════
st.markdown("---")
st.subheader("🔮 Live Prediction Panel")
st.markdown("🎛️ **Adjust any slider below — predictions update automatically in real time.**")

def enc_season(m):
    """Convert month number → (season integer code, season name string)"""
    s = ('Long_Rains'  if m in [3,4,5]   else
         'Dry'         if m in [6,7,8]   else
         'Short_Rains' if m in [9,10,11] else 'Cool_Dry')
    return {'Cool_Dry':0,'Dry':1,'Long_Rains':2,'Short_Rains':3}[s], s


# ── INPUT WIDGETS ─────────────────────────────────────────────
# Each slider has a unique key= argument. This is REQUIRED —
# Streamlit uses keys to track which widget changed and to
# store values across reruns without resetting to defaults.
# ─────────────────────────────────────────────────────────────
st.markdown("#### 🌦️ Climate Conditions")
p1, p2, p3 = st.columns(3)
rainfall    = p1.slider("🌧️ Rainfall (mm)",     0.0, 350.0, 150.0, 5.0,  key="rf")
temperature = p2.slider("🌡️ Temperature (°C)", 10.0,  40.0,  24.0, 0.5,  key="tmp")
humidity    = p3.slider("💧 Humidity (%)",       0.0, 100.0,  70.0, 1.0,  key="hum")

st.markdown("#### 🦟 Epidemiological Inputs")
p4, p5, p6 = st.columns(3)
lag_cases   = p4.slider("📅 Lag 1 Month Cases",  0.0, 3500.0, 1200.0, 50.0, key="lag")
incidence   = p5.slider("📊 Incidence per 100k", 0.0,  600.0,  120.0,  5.0, key="inc")
month       = p6.selectbox("📆 Month", list(range(1,13)),
                            format_func=lambda m:MN[m-1], index=5, key="mon")

st.markdown("#### 👥 Population")
p7, p8 = st.columns(2)
population  = p7.slider("🏘️ Population",    100000, 3000000, 500000, 10000, key="pop")
mal_cases   = p8.slider("🦟 Malaria Cases",    0.0,  3500.0, 1200.0,  50.0, key="mc")


# ══════════════════════════════════════════════════════════════════
# SECTION 15 — LIVE PREDICTION COMPUTATION
# Runs on every slider change (no button needed).
#
#   1. Derive season code and cases-per-capita from slider values
#   2. Build a single-row DataFrame matching the 12 training features
#   3. Load the scaler from disk → apply only to Logistic Regression
#      (tree models don't need scaling)
#   4. Load each saved model from disk → predict class + probability
#   5. Display coloured risk cards for each model
#   6. Plot the live probability bar chart
#   7. Plot the ensemble gauge bar (average probability of all models)
#   8. Show a 4-metric input summary with High/Normal labels
# ══════════════════════════════════════════════════════════════════

# Step 1: Derive engineered features from current slider values
se, sn = enc_season(month)
cpc    = mal_cases / max(population, 1) * 100000   # cases per capita

# Step 2: Build one-row input DataFrame aligned to training features
inp = pd.DataFrame(
    [[rainfall, temperature, humidity, lag_cases, incidence,
      month, population, mal_cases, cpc, 0, 0, se]],
    columns=FEATURES
)

# Step 3: Load the scaler that was fitted during training
sc_loaded = joblib.load(f"{SAVE_DIR}/scaler.pkl")

st.markdown("---")
st.markdown("### 🎯 Live Prediction Results")
st.caption(f"📍 Season: **{sn.replace('_',' ')}**  |  Cases per capita: **{cpc:.1f}** per 100k")

# Step 4 & 5: Load each model → predict → display risk card
probs = []
rcols = st.columns(len(trained))
for i, name in enumerate(trained):
    # Load model from disk (survives every Streamlit rerun)
    m    = joblib.load(f"{SAVE_DIR}/{name.replace(' ','_')}.pkl")
    # Scale only for Logistic Regression
    X_in = sc_loaded.transform(inp) if name == "Logistic Regression" else inp
    pred = m.predict(X_in)[0]            # 0 = Low Risk, 1 = High Risk
    prob = m.predict_proba(X_in)[0][1]   # probability of being High Risk
    probs.append(prob)
    bg  = '#c0392b' if pred == 1 else '#117a65'   # red or green card
    lbl = "🔴 HIGH RISK" if pred == 1 else "🟢 LOW RISK"
    rcols[i].markdown(f"""
    <div style="background:{bg};padding:1.4rem;border-radius:12px;
                text-align:center;color:white;margin:4px;
                box-shadow:0 2px 8px rgba(0,0,0,0.2);">
        <div style="font-size:0.85rem;opacity:0.85;margin-bottom:4px;">{name}</div>
        <div style="font-size:1.6rem;font-weight:bold;margin:0.3rem 0;">{lbl}</div>
        <div style="font-size:1.1rem;">Confidence: <b>{prob*100:.1f}%</b></div>
    </div>""", unsafe_allow_html=True)

# Ensemble average — mean probability across all selected models
avg = np.mean(probs)

# Step 6: Live probability bar chart — one bar per model
fig, ax = plt.subplots(figsize=(8,4))
bar_colors = ['#c0392b' if p >= 0.5 else '#117a65' for p in probs]
bars = ax.bar(trained, [p*100 for p in probs],
              color=bar_colors, edgecolor='white', alpha=0.9, width=0.45)
ax.axhline(50, color='grey', lw=1.5, linestyle='--', alpha=0.7,
           label='50% Risk Threshold')
for bar, p in zip(bars, probs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
            f'{p*100:.1f}%', ha='center', va='bottom',
            fontsize=13, fontweight='bold',
            color='#c0392b' if p >= 0.5 else '#117a65')
ax.set_ylim(0,120)
ax.set_ylabel('High-Risk Probability (%)', fontsize=11)
ax.set_title(f'Live Risk Probability  |  Ensemble Average: {avg*100:.1f}%',
             fontweight='bold', fontsize=12)
ax.legend(fontsize=10); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); st.pyplot(fig); plt.close()

# Step 7: Gauge bar — shows ensemble average as a horizontal bar
risk_label = "🔴 HIGH RISK" if avg >= 0.5 else "🟢 LOW RISK"
gauge_col  = '#c0392b'      if avg >= 0.5 else '#117a65'
fig, ax = plt.subplots(figsize=(10,1.5))
ax.barh([0],[1], color='#ecf0f1', height=0.6)            # full grey background
ax.barh([0],[avg], color=gauge_col, height=0.6, alpha=0.9) # coloured fill
ax.axvline(0.5, color='#555', lw=2.5, linestyle='--')    # threshold line
ax.set_xlim(0,1); ax.set_yticks([])
ax.set_xticks([0,0.25,0.5,0.75,1.0])
ax.set_xticklabels(['0%','25%','50% threshold','75%','100%'], fontsize=10)
ax.set_title(f'Ensemble Risk Score: {avg*100:.1f}%   →   {risk_label}',
             fontweight='bold', fontsize=13, color=gauge_col)
ax.spines[['top','right','left']].set_visible(False)
plt.tight_layout(); st.pyplot(fig); plt.close()

# Step 8: Input summary metric cards with High/Normal indicator
st.markdown("#### 📋 Input Summary")
s1, s2, s3, s4 = st.columns(4)
s1.metric("🌧️ Rainfall",    f"{rainfall:.0f} mm",   delta="High ⚠️"   if rainfall    > 200  else "Normal ✅")
s2.metric("🌡️ Temperature", f"{temperature:.1f}°C", delta="High ⚠️"   if temperature > 30   else "Normal ✅")
s3.metric("💧 Humidity",    f"{humidity:.0f}%",      delta="High ⚠️"   if humidity    > 80   else "Normal ✅")
s4.metric("🦟 Lag Cases",   f"{lag_cases:.0f}",      delta="High ⚠️"   if lag_cases   > 1500 else "Normal ✅")


# ══════════════════════════════════════════════════════════════════
# SECTION 16 — FOOTER
# st.caption() renders small grey text — used for attribution.
# ══════════════════════════════════════════════════════════════════
st.markdown("---")
st.caption("Group 3 | BSc Data Science | Meru University of Science and Technology | 2026")


Overwriting malaria_app.py


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║              CELL 3 — LAUNCH STREAMLIT + TUNNEL                 ║
# ║  This cell starts the Streamlit web server as a background      ║
# ║  process, then opens a public SSH tunnel so anyone can          ║
# ║  access the app from a browser link.                            ║
# ║          ║
# ╚══════════════════════════════════════════════════════════════════╝


import subprocess, time, socket, re, os, fcntl

PORT = 8501

# ── STEP 1: Kill any previous Streamlit process ───────────────────
print("🛑 Stopping any old Streamlit...")
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(3)   # wait for port to be fully released

# ── STEP 2: Start Streamlit as a background subprocess ───────────
print("🚀 Starting Streamlit...")
subprocess.Popen(
    ["streamlit", "run", "malaria_app.py",
     "--server.port",                 str(PORT),
     "--server.headless",             "true",
     "--browser.gatherUsageStats",    "false",
     "--server.enableCORS",           "false",
     "--server.enableXsrfProtection", "false"],
    stdout=open("st.log",     "w"),
    stderr=open("st_err.log", "w")
)

# ── STEP 3: Wait until Streamlit port is actually open ────────────
# Instead of a fixed sleep, we poll every 2 seconds for up to 60s.
# This way the tunnel only opens AFTER Streamlit is truly ready.
def port_open(p):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", p)) == 0

print(f"⏳ Waiting for Streamlit on port {PORT}...", end="")
for i in range(30):          # try for up to 60 seconds (30 x 2s)
    if port_open(PORT):
        print(f" ready after {(i+1)*2}s ✅")
        break
    print(".", end="", flush=True)
    time.sleep(2)
else:
    print("\n❌ Streamlit did not start. Error log:")
    print(open("st_err.log").read())
    raise SystemExit("Fix the error above, then re-run this cell.")

# Extra buffer — give Streamlit 5 more seconds to finish loading
# all its internal components before accepting tunnel traffic
time.sleep(5)

# ── STEP 4: Open SSH tunnel via localhost.run ─────────────────────
# Kill any old tunnel first
subprocess.run(["pkill", "-f", "localhost.run"], capture_output=True)
subprocess.run(["pkill", "-f", "serveo.net"],    capture_output=True)
time.sleep(2)

print("🌐 Opening public tunnel...")
tunnel = subprocess.Popen(
    ["ssh",
     "-o", "StrictHostKeyChecking=no",
     "-o", "ServerAliveInterval=60",   # keep-alive pings every 60s
     "-o", "ServerAliveCountMax=10",   # allow 10 missed pings
     "-R", f"80:localhost:{PORT}",
     "nokey@localhost.run"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Make pipe non-blocking so we can read without hanging
fcntl.fcntl(tunnel.stdout, fcntl.F_SETFL, os.O_NONBLOCK)

# Poll for the URL — check every 2 seconds for up to 40 seconds
output = ""
link_found = False
for attempt in range(20):
    time.sleep(2)
    try:
        chunk = tunnel.stdout.read(8192)
        if chunk:
            output += chunk.decode("utf-8", errors="ignore")
    except:
        pass

    urls = re.findall(r'https://[a-zA-Z0-9\-]+\.lhr\.life', output)
    if urls:
        link_found = True
        print(f"\n{'='*58}")
        print(f"  🚀  YOUR LIVE LINK  →  {urls[0]}")
        print(f"{'='*58}")
        print("  Steps:")
        print("  1️⃣  Open the link above in your browser")
        print("  2️⃣  Upload your CSV in the LEFT sidebar")
        print("  3️⃣  Click 🚀 Train Models")
        print("  4️⃣  Move sliders — predictions update live!")
        print(f"\n  ⚠️  Keep this Colab tab open while using the app.")
        break

if not link_found:
    # ── FALLBACK: serveo.net ──────────────────────────────────────
    print("localhost.run slow — trying serveo.net...")
    tunnel.kill()
    time.sleep(2)

    tunnel2 = subprocess.Popen(
        ["ssh",
         "-o", "StrictHostKeyChecking=no",
         "-o", "ServerAliveInterval=60",
         "-R", f"80:localhost:{PORT}",
         "serveo.net"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    fcntl.fcntl(tunnel2.stdout, fcntl.F_SETFL, os.O_NONBLOCK)

    output2 = ""
    for attempt in range(20):
        time.sleep(2)
        try:
            chunk = tunnel2.stdout.read(8192)
            if chunk:
                output2 += chunk.decode("utf-8", errors="ignore")
        except:
            pass

        urls2 = re.findall(r'https?://[a-zA-Z0-9\-]+\.serveo\.net', output2)
        if urls2:
            print(f"\n{'='*58}")
            print(f"  🚀  YOUR LIVE LINK  →  {urls2[0]}")
            print(f"{'='*58}")
            print("  1️⃣  Open the link in your browser")
            print("  2️⃣  Upload CSV → Train Models → move sliders!")
            break
    else:
        print("\n❌ Both tunnel services failed.")
        print("👉 Try this manual fix in Colab:")
        print("   Runtime menu → Ports → Add port 8501")
        print("\n── Streamlit log (for debugging) ──")
        print(open("st_err.log").read())


🛑 Stopping any old Streamlit...
🚀 Starting Streamlit...
⏳ Waiting for Streamlit on port 8501..... ready after 6s ✅
🌐 Opening public tunnel...

  🚀  YOUR LIVE LINK  →  https://290c13499ff705.lhr.life
  Steps:
  1️⃣  Open the link above in your browser
  2️⃣  Upload your CSV in the LEFT sidebar
  3️⃣  Click 🚀 Train Models
  4️⃣  Move sliders — predictions update live!

  ⚠️  Keep this Colab tab open while using the app.
